In [ ]:
!pip install langchain langchain-community faiss-cpu sentence-transformers groq ragas datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.6 MB/s eta 0:00:00


In [ ]:
import os

os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model="llama-3.1-8b-instant"   # fast and good for RAG
)

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(page_content="RAG stands for Retrieval Augmented Generation. It improves LLM responses by using external knowledge."),
    Document(page_content="FAISS is a vector database used for efficient similarity search of embeddings."),
    Document(page_content="Groq provides fast inference for large language models like LLaMA 3."),
    Document(page_content="Embeddings convert text into numerical vectors for semantic search.")
]

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(docs, embeddings)

In [ ]:
query = "What is RAG?"
results = vector_db.similarity_search(query)

for r in results:
    print(r.page_content)

RAG stands for Retrieval Augmented Generation. It improves LLM responses by using external knowledge.
FAISS is a vector database used for efficient similarity search of embeddings.
Groq provides fast inference for large language models like LLaMA 3.
Embeddings convert text into numerical vectors for semantic search.


In [ ]:
retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
query = "Explain embeddings"
docs_found = retriever.invoke(query)

for doc in docs_found:
    print(doc.page_content)
    print("-----")

FAISS is a vector database used for efficient similarity search of embeddings.
-----
Embeddings convert text into numerical vectors for semantic search.
-----
Groq provides fast inference for large language models like LLaMA 3.
-----


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an AI assistant. Use the given context to answer the question.

Context:
{context}

Question:
{question}

Answer clearly and accurately:
"""
)

In [ ]:
!pip install -U langchain langchain-core langchain-community langchain-groq

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant. Use the context below to answer the question.

Context:
{context}

Question:
{input}

Answer clearly:
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {
        "context": (lambda x: x["input"]) | retriever | format_docs,
        "input": lambda x: x["input"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
response = rag_chain.invoke({"input": "What is RAG?"})
print(response)

RAG stands for Retrieval Augmented Generation. It is a method that improves the responses of Large Language Models (LLMs) by utilizing external knowledge. RAG retrieves relevant information from a database and incorporates it into the LLM's response, making the output more accurate and informative.


In [ ]:
question = "What is RAG?"

answer = rag_chain.invoke({"input": question})

In [ ]:
retrieved_docs = retriever.invoke(question)

context = [doc.page_content for doc in retrieved_docs]

In [ ]:
eval_data = {
    "question": question,
    "answer": answer,
    "contexts": context,
    "ground_truth": "Retrieval Augmented Generation combines LLMs with external knowledge sources to improve responses."
}

In [ ]:
dataset = [eval_data]

print(dataset)

[{'question': 'What is RAG?', 'answer': "RAG stands for Retrieval Augmented Generation. It's a technique used to improve the responses of Large Language Models (LLMs) by incorporating external knowledge.", 'contexts': ['RAG stands for Retrieval Augmented Generation. It improves LLM responses by using external knowledge.', 'FAISS is a vector database used for efficient similarity search of embeddings.', 'Groq provides fast inference for large language models like LLaMA 3.'], 'ground_truth': 'Retrieval Augmented Generation combines LLMs with external knowledge sources to improve responses.'}]


In [ ]:
!pip install ragas datasets

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision


/tmp/ipykernel_567/3069169274.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_567/3069169274.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/tmp/ipykernel_567/3069169274.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, answer_relevancy, context_preci

In [ ]:
data = {
    "question": [eval_data["question"]],
    "answer": [eval_data["answer"]],
    "contexts": [eval_data["contexts"]],
    "ground_truth": [eval_data["ground_truth"]]
}

dataset = Dataset.from_dict(data)

In [ ]:
results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision]
)

Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: InstructorRetryException(<failed_attempts>

<generation number="1">
<exception>
    Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="3">
<exception>
    Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type'

In [ ]:
print(results)

{'faithfulness': nan, 'answer_relevancy': nan, 'context_precision': nan}
